<a href="https://colab.research.google.com/github/rishithareddym/AI-Lab/blob/main/1BF24CS181_WEEK9_RESOLUTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re

def is_variable(term):
    """Variables start with a lowercase letter (e.g., x, y, z)."""
    return isinstance(term, str) and term[0].islower() and '(' not in term

def parse_literal(literal):
    """Parses '~Food(x)' into (True, 'Food', ['x'])."""
    negated = literal.startswith('~')
    clean = literal[1:] if negated else literal
    match = re.match(r"([A-Za-z0-9_]+)\((.*)\)", clean)
    if match:
        pred = match.group(1)
        args = [arg.strip() for arg in match.group(2).split(',')]
        return negated, pred, args
    return negated, clean, []

def format_literal(negated, pred, args):
    """Formats (True, 'Food', ['x']) back into '~Food(x)'."""
    prefix = '~' if negated else ''
    return f"{prefix}{pred}({', '.join(args)})" if args else f"{prefix}{pred}"

def unify(x, y, subs=None):
    """Unifies two terms or argument lists and returns a substitution dictionary."""
    if subs is None:
        subs = {}
    if subs is None:
        return None
    if x == y:
        return subs
    if is_variable(x):
        return unify_var(x, y, subs)
    if is_variable(y):
        return unify_var(y, x, subs)
    if isinstance(x, list) and isinstance(y, list) and len(x) == len(y):
        for a, b in zip(x, y):
            subs = unify(a, b, subs)
            if subs is None:
                return None
        return subs
    return None

def unify_var(var, x, subs):
    if var in subs:
        return unify(subs[var], x, subs)
    if x in subs:
        return unify(var, subs[x], subs)
    # Occurs check
    if isinstance(x, list) and var in x:
        return None
    new_subs = subs.copy()
    new_subs[var] = x
    return new_subs

def apply_subs(clause, subs):
    """Applies variable substitutions to all literals in a clause."""
    new_clause = []
    for lit in clause:
        neg, pred, args = parse_literal(lit)
        new_args = []
        for arg in args:
            val = arg
            while val in subs:
                val = subs[val]
            new_args.append(val)
        new_clause.append(format_literal(neg, pred, new_args))
    return frozenset(new_clause)

def standardize_variables(clause, counter):
    """Renames variables in a clause so they don't clash with other clauses."""
    mapping = {}
    new_clause = []
    for lit in clause:
        neg, pred, args = parse_literal(lit)
        new_args = []
        for arg in args:
            if is_variable(arg):
                if arg not in mapping:
                    mapping[arg] = f"{arg}_{counter}"
                new_args.append(mapping[arg])
            else:
                new_args.append(arg)
        new_clause.append(format_literal(neg, pred, new_args))
    return frozenset(new_clause)

def resolve(c1, c2):
    """Resolves two FOL clauses using unification."""
    resolvents = []
    for lit1 in c1:
        neg1, pred1, args1 = parse_literal(lit1)
        for lit2 in c2:
            neg2, pred2, args2 = parse_literal(lit2)
            # Check for opposite signs and matching predicate names
            if pred1 == pred2 and neg1 != neg2:
                subs = unify(args1, args2, {})
                if subs is not None:
                    rem1 = [l for l in c1 if l != lit1]
                    rem2 = [l for l in c2 if l != lit2]
                    combined = apply_subs(rem1 + rem2, subs)
                    resolvents.append((combined, subs))
    return resolvents

def fol_resolution(kb_clauses, query):
    """Main FOL Resolution Refutation loop."""
    neg_query = query[1:] if query.startswith('~') else f"~{query}"
    clauses = [frozenset(c) for c in kb_clauses]
    clauses.append(frozenset([neg_query]))

    seen = set(clauses)
    var_counter = 1
    step = 1

    print("\n--- INITIAL CLAUSES IN CNF ---")
    for idx, c in enumerate(clauses, 1):
        print(f"C{idx}: {set(c)}")
    print("-" * 50)

    while True:
        new_clauses = []
        n = len(clauses)
        for i in range(n):
            for j in range(i + 1, n):
                c1 = standardize_variables(clauses[i], var_counter)
                var_counter += 1
                c2 = standardize_variables(clauses[j], var_counter)
                var_counter += 1

                for res, subs in resolve(c1, c2):
                    if len(res) == 0:
                        print(f"Step {step}: Resolve {set(clauses[i])} & {set(clauses[j])} with {subs} -> EMPTY CLAUSE {{}}")
                        return True
                    if res not in seen:
                        print(f"Step {step}: Resolve {set(clauses[i])} & {set(clauses[j])} with {subs} -> {set(res)}")
                        step += 1
                        seen.add(res)
                        new_clauses.append(res)

        if not new_clauses:
            return False
        clauses.extend(new_clauses)

# --- USER INPUT SECTION ---
if __name__ == "__main__":
    print("=== FOL RESOLUTION PROVER (CNF) ===")
    print("Rules: Use '~' for NOT, '|' for OR. Variables = lowercase (x, y), Constants/Predicates = Uppercase (John, Food).\n")

    num = int(input("Enter number of KB clauses: "))
    kb = []
    for i in range(num):
        raw_clause = input(f"Enter Clause {i+1}: ")
        literals = [lit.strip() for lit in raw_clause.split('|')]
        kb.append(literals)

    goal = input("\nEnter the Goal/Conclusion to prove: ").strip()

    if fol_resolution(kb, goal):
        print(f"\nRESULT: Contradiction reached! '{goal}' is PROVED.")
    else:
        print(f"\nRESULT: Cannot prove '{goal}' from the given KB.")


=== FOL RESOLUTION PROVER (CNF) ===
Rules: Use '~' for NOT, '|' for OR. Variables = lowercase (x, y), Constants/Predicates = Uppercase (John, Food).

Enter number of KB clauses: 5
Enter Clause 1: ~Food(x) | Likes(John, x)
Enter Clause 2:  Food(Apple)
Enter Clause 3:  ~Eats(y, z) | Killed(y) | Food(z)
Enter Clause 4:  Eats(Bill, Peanuts)
Enter Clause 5: ~Killed(Bill)

Enter the Goal/Conclusion to prove: Likes(John, Peanuts)

--- INITIAL CLAUSES IN CNF ---
C1: {'~Food(x)', 'Likes(John, x)'}
C2: {'Food(Apple)'}
C3: {'~Eats(y, z)', 'Food(z)', 'Killed(y)'}
C4: {'Eats(Bill, Peanuts)'}
C5: {'~Killed(Bill)'}
C6: {'~Likes(John, Peanuts)'}
--------------------------------------------------
Step 1: Resolve {'~Food(x)', 'Likes(John, x)'} & {'Food(Apple)'} with {'x_1': 'Apple'} -> {'Likes(John, Apple)'}
Step 2: Resolve {'~Food(x)', 'Likes(John, x)'} & {'~Eats(y, z)', 'Food(z)', 'Killed(y)'} with {'x_3': 'z_4'} -> {'Likes(John, z_4)', 'Killed(y_4)', '~Eats(y_4, z_4)'}
Step 3: Resolve {'~Food(x)', 'L